# Amazon SageMaker

A practical reference for **Amazon SageMaker** — AWS's fully managed platform for the end-to-end
machine-learning lifecycle: label, build, train, tune, deploy, and monitor models without
managing the underlying training clusters or inference servers yourself.

## Table of Contents

1. [Introduction](#introduction)
2. [Key Features](#key-features)
3. [Architecture Overview](#architecture)
4. [Installation](#installation)
5. [Basic Usage](#basic-usage)
6. [Advanced Features](#advanced-features)
7. [Use Cases](#use-cases)
8. [Best Practices](#best-practices)
9. [Common Pitfalls](#pitfalls)
10. [Performance Optimization](#performance)
11. [Production Deployment](#deployment)
12. [Monitoring and Observability](#monitoring)
13. [Troubleshooting](#troubleshooting)
14. [Comparison with Alternatives](#comparison)
15. [Resources](#resources)

## Introduction

<a id='introduction'></a>

### What is it?

**Amazon SageMaker** is a managed service that removes the undifferentiated heavy lifting from
machine learning. Instead of provisioning GPU boxes, wiring up distributed training, and
running your own model server, you describe *what* you want — "train this estimator on this S3
data with 4 `ml.p4d.24xlarge` instances" or "deploy this model as a real-time endpoint" — and
SageMaker spins up the infrastructure, runs the job, ships logs and metrics to CloudWatch, and
tears everything down when it's done. You pay per second for the instances while they run.

SageMaker is an umbrella over many sub-services: **Studio** (a web IDE), **Training Jobs**,
**Processing Jobs**, **Hyperparameter Tuning**, several **Inference** options, a **Model
Registry**, **Pipelines** (CI/CD for ML), **Feature Store**, **Clarify** (bias/explainability),
**Model Monitor** (drift detection), and **JumpStart** (a hub of pretrained/foundation models).

### Why use it?

- **No cluster to babysit.** Training and inference infrastructure is created on demand and
  billed per second; nothing idles between jobs.
- **Managed distributed training.** Built-in data-parallel (SMDDP) and model-parallel (SMP)
  libraries, plus first-class support for `torchrun`/MPI, make multi-GPU/multi-node training a
  config change rather than a project.
- **Prebuilt Deep Learning Containers (DLCs).** Optimized PyTorch/TensorFlow/HuggingFace/XGBoost
  images are maintained by AWS, so you usually bring only a training script.
- **Full lifecycle in one place.** Data processing, training, tuning, registry, deployment,
  monitoring, and pipeline orchestration share IAM, S3, and lineage tracking.
- **Many deployment shapes.** Real-time, serverless, asynchronous, and batch inference cover
  everything from low-latency APIs to multi-GB offline scoring.

### When to use it?

- You're on AWS and want managed training/serving rather than running your own Kubernetes +
  Kubeflow stack.
- You need to scale to multi-node GPU training occasionally without owning the hardware.
- You want managed endpoints with autoscaling, A/B traffic shifting, and drift monitoring.
- You want a reproducible ML pipeline (data → train → evaluate → register → deploy) wired into
  IAM and S3.

It is **less** of a fit when you need fine-grained control over the cluster, are multi-cloud, or
your workloads are so steady that reserved self-managed hardware is cheaper.

## Key Features

<a id='key-features'></a>

### Core Capabilities of Amazon SageMaker

| Feature | Description | Benefit |
|---------|-------------|---------|
| Training Jobs | Ephemeral, fully managed training clusters fed from S3 | GPU clusters on demand, billed per second, auto-torn-down |
| Framework estimators | `PyTorch`, `HuggingFace`, `TensorFlow`, `XGBoost`, `SKLearn` wrappers over AWS DLCs | Bring a script, not a Dockerfile |
| Managed Spot Training | Run on spare capacity with checkpoint/resume | Up to ~90% lower training cost |
| Distributed libraries | SMDDP (data parallel) and SMP (tensor/pipeline parallel) | Multi-node scaling as a config flag |
| Inference options | Real-time, Serverless, Asynchronous, Batch Transform | One model, four serving shapes for different SLAs |
| Automatic Model Tuning | Bayesian / Hyperband hyperparameter search | Better models without a hand-rolled sweep harness |
| Pipelines + Model Registry | DAG orchestration with versioned, approvable models | Reproducible, gated path to production |
| Feature Store / Clarify / Model Monitor | Shared features, bias/explainability, drift detection | Governance and observability built in |

## Architecture Overview

<a id='architecture'></a>

```
                          Your code (SageMaker Python SDK / boto3)
                                         |
              estimator.fit()            |            estimator.deploy()
        +------------------------+       |       +---------------------------+
        |   Training Job         |       |       |   Inference Endpoint      |
        |  (ephemeral cluster)   |       |       |  (long-lived autoscaling) |
        |                        |       |       |                           |
        |  ml.p4d.24xlarge x N   |       |       |  ml.g5.2xlarge x M        |
        |  + AWS DLC image       |       |       |  + model.tar.gz           |
        +-----------+------------+               +-------------+-------------+
                    |  reads input channels                    |  writes preds
                    v                                          v
        +------------------------------------------------------------------+
        |                         Amazon S3                                 |
        |   s3://bucket/data/   s3://bucket/model/   s3://bucket/output/    |
        +------------------------------------------------------------------+
                    |                                          |
              CloudWatch Logs / Metrics             IAM execution role (S3, ECR, logs)
```

### Components

1. **SageMaker Python SDK / boto3** — your control plane. The high-level `sagemaker` SDK gives
   `Estimator`, `Predictor`, `Processor`, `Pipeline`; boto3's `sagemaker` client exposes the
   raw `CreateTrainingJob`/`CreateEndpoint` API underneath.
2. **Training Job** — an ephemeral cluster SageMaker creates, runs your container on, then
   destroys. Input data is pulled from **S3 channels**; the trained model is written back to S3
   as `model.tar.gz`.
3. **Deep Learning Containers (DLCs)** — AWS-maintained ECR images with framework + CUDA +
   SageMaker toolkit baked in; framework estimators pick the right one automatically.
4. **Inference Endpoint** — a long-lived, autoscaling HTTPS service hosting your model
   (real-time), or one of the serverless/async/batch variants.
5. **Amazon S3** — the data backbone: training inputs, model artifacts, and outputs all live in
   S3 and are referenced by URI.
6. **IAM execution role** — the identity the jobs/endpoints assume to read S3, pull from ECR,
   and write CloudWatch logs. Nearly every "AccessDenied" issue traces back here.

## Installation

<a id='installation'></a>

### Prerequisites

- An **AWS account** with permissions to use SageMaker, and an **IAM execution role** that
  SageMaker can assume (typically with `AmazonSageMakerFullAccess` + S3 access).
- **AWS credentials** configured locally (`aws configure`) or, inside SageMaker Studio, the
  ambient Studio role.
- Python 3.8+.

### Installation Steps

The SageMaker Python SDK plus boto3 are all you need on the client side; the heavy frameworks
run inside AWS-managed containers, not locally.

In [ ]:
# Uncomment to install the SageMaker Python SDK and AWS SDK locally.
# Inside SageMaker Studio these are preinstalled.
# !pip install -U sagemaker boto3

import sagemaker
import boto3

print("sagemaker SDK:", sagemaker.__version__)
print("boto3:", boto3.__version__)

# A Session ties SDK calls to a region + default S3 bucket.
# (This call needs real AWS credentials; shown here as the standard entry point.)
# session = sagemaker.Session()
# print("region:", session.boto_region_name)
# print("default bucket:", session.default_bucket())

## Basic Usage

<a id='basic-usage'></a>

### Quick Start Example

The canonical flow is **estimator → fit → deploy → predict**. A framework estimator wraps your
training script and the right AWS DLC image; `fit()` launches a managed training job; `deploy()`
turns the resulting artifact into a real-time endpoint.

```python
from sagemaker.pytorch import PyTorch

estimator = PyTorch(
    entry_point="train.py",        # your script
    source_dir="src",              # extra modules / requirements.txt
    role=role,                     # IAM execution role ARN
    instance_type="ml.g5.2xlarge", # one A10G GPU
    instance_count=1,
    framework_version="2.2",
    py_version="py310",
    hyperparameters={"epochs": 10, "lr": 1e-3},
)

# Input channels map names to S3 prefixes; they appear as local dirs in the container,
# discoverable via the SM_CHANNEL_TRAIN environment variable.
estimator.fit({"train": "s3://my-bucket/data/train",
               "val":   "s3://my-bucket/data/val"})

# Deploy the trained artifact as an autoscaling HTTPS endpoint.
predictor = estimator.deploy(initial_instance_count=1,
                             instance_type="ml.g5.xlarge")

result = predictor.predict({"inputs": "hello"})
predictor.delete_endpoint()        # ALWAYS clean up — endpoints bill per second
```

Inside `train.py`, SageMaker hands you the contract via environment variables:

```python
# train.py (runs inside the SageMaker container)
import os, argparse

parser = argparse.ArgumentParser()
parser.add_argument("--epochs", type=int, default=int(os.environ.get("SM_HP_EPOCHS", 1)))
parser.add_argument("--train", default=os.environ["SM_CHANNEL_TRAIN"])   # local path
parser.add_argument("--model-dir", default=os.environ["SM_MODEL_DIR"])    # save here -> S3
args = parser.parse_args()
# ... train, then torch.save(model, os.path.join(args.model_dir, "model.pt"))
```

In [ ]:
# Construct an estimator's job config WITHOUT calling AWS, so the contract is inspectable
# offline. This is exactly the request the SDK would send to CreateTrainingJob.
def training_job_request(entry_point, role, instance_type, instance_count,
                         image_uri, input_channels, hyperparameters,
                         use_spot=False, max_run=3600):
    cfg = {
        "AlgorithmSpecification": {
            "TrainingImage": image_uri,
            "TrainingInputMode": "File",          # File | FastFile | Pipe
        },
        "RoleArn": role,
        "ResourceConfig": {
            "InstanceType": instance_type,
            "InstanceCount": instance_count,
            "VolumeSizeInGB": 50,
        },
        "HyperParameters": {k: str(v) for k, v in hyperparameters.items()} |
                           {"sagemaker_program": entry_point},
        "InputDataConfig": [
            {"ChannelName": name,
             "DataSource": {"S3DataSource": {"S3Uri": uri, "S3DataType": "S3Prefix"}}}
            for name, uri in input_channels.items()
        ],
        "OutputDataConfig": {"S3OutputPath": "s3://my-bucket/output/"},
        "StoppingCondition": {"MaxRuntimeInSeconds": max_run},
    }
    if use_spot:
        cfg["EnableManagedSpotTraining"] = True
        cfg["StoppingCondition"]["MaxWaitTimeInSeconds"] = max_run * 2
    return cfg

import json
req = training_job_request(
    entry_point="train.py",
    role="arn:aws:iam::123456789012:role/SageMakerExecutionRole",
    instance_type="ml.g5.2xlarge", instance_count=1,
    image_uri="763104351884.dkr.ecr.us-east-1.amazonaws.com/pytorch-training:2.2-gpu-py310",
    input_channels={"train": "s3://my-bucket/data/train"},
    hyperparameters={"epochs": 10, "lr": 1e-3},
    use_spot=True,
)
print(json.dumps(req, indent=2))

## Advanced Features

<a id='advanced-features'></a>

### Managed Spot Training

Spot training runs on spare EC2 capacity for up to ~90% less than on-demand. Because Spot
instances can be reclaimed, you set a `max_wait` (≥ `max_run`) and checkpoint to S3 so an
interrupted job **resumes** instead of restarting:

```python
estimator = PyTorch(
    ..., use_spot_instances=True,
    max_run=3600, max_wait=7200,
    checkpoint_s3_uri="s3://my-bucket/checkpoints/job-x",  # synced to /opt/ml/checkpoints
)
```

### Distributed training

- **Data parallel (SMDDP):** replicate the model, shard the batch. Enable with
  `distribution={"smdistributed": {"dataparallel": {"enabled": True}}}` or just
  `distribution={"torch_distributed": {"enabled": True}}` for native `torchrun`.
- **Model parallel (SMP v2):** shard parameters/optimizer state (FSDP-style) and add
  tensor/pipeline parallelism for models too big for one GPU.

```python
estimator = PyTorch(
    ..., instance_type="ml.p4d.24xlarge", instance_count=4,
    distribution={"torch_distributed": {"enabled": True}},  # 32 GPUs, torchrun-managed
)
```

### Automatic Model Tuning

Define hyperparameter ranges and a metric to optimize; SageMaker runs many training jobs with
Bayesian or Hyperband search:

```python
from sagemaker.tuner import HyperparameterTuner, ContinuousParameter, IntegerParameter

tuner = HyperparameterTuner(
    estimator,
    objective_metric_name="val:accuracy",
    metric_definitions=[{"Name": "val:accuracy", "Regex": "val_acc=([0-9\.]+)"}],
    hyperparameter_ranges={"lr": ContinuousParameter(1e-5, 1e-2),
                           "batch_size": IntegerParameter(16, 128)},
    max_jobs=20, max_parallel_jobs=4, strategy="Bayesian",
)
tuner.fit({"train": "s3://my-bucket/data/train"})
```

### Pipelines + Model Registry

`sagemaker.workflow` builds a DAG of `ProcessingStep → TrainingStep → RegisterModel` so the path
to production is versioned and gated by a manual/automated **approval** in the Model Registry,
which then triggers deployment.

In [ ]:
# Pick the cheapest deployment shape for a workload from its traffic profile.
# Real-time | Serverless | Async | Batch are SageMaker's four inference options.
def choose_inference_option(qps, p99_latency_budget_ms, payload_mb, traffic="steady"):
    if traffic == "offline":
        return ("Batch Transform",
                "No live endpoint; score a whole S3 dataset, cluster torn down after.")
    if payload_mb > 6 or p99_latency_budget_ms > 60_000:
        return ("Asynchronous Inference",
                "Queues large/long requests to S3; scales to zero when idle.")
    if traffic == "spiky" and qps < 50:
        return ("Serverless Inference",
                "No instance management; pay per request; tolerates cold starts.")
    return ("Real-time Endpoint",
            "Always-on, low-latency, autoscaling instances for steady high QPS.")

for case in [
    dict(qps=500, p99_latency_budget_ms=50,    payload_mb=0.1, traffic="steady"),
    dict(qps=5,   p99_latency_budget_ms=200,   payload_mb=0.1, traffic="spiky"),
    dict(qps=2,   p99_latency_budget_ms=120000, payload_mb=80, traffic="steady"),
    dict(qps=0,   p99_latency_budget_ms=0,     payload_mb=500, traffic="offline"),
]:
    name, why = choose_inference_option(**case)
    print(f"{case}\n  -> {name}: {why}\n")

## Use Cases

<a id='use-cases'></a>

#### Use Case 1: Fine-tuning an LLM on managed Spot GPUs

- **Context:** Fine-tune a 7B-parameter model on domain data; you don't own A100/H100 hardware
  and the job runs for a few hours occasionally.
- **Implementation:** `HuggingFace` estimator on `ml.p4d.24xlarge` with
  `use_spot_instances=True`, `checkpoint_s3_uri` for resume, and `torch_distributed` across 8
  GPUs. Data streamed from S3 with `TrainingInputMode="FastFile"`.
- **Result:** Multi-GPU fine-tuning at a fraction of on-demand cost, with no standing cluster.

#### Use Case 2: Real-time fraud scoring API

- **Context:** A transaction service needs <50 ms p99 model scoring at thousands of QPS.
- **Implementation:** XGBoost model deployed to a **real-time endpoint** with target-tracking
  autoscaling on `SageMakerVariantInvocationsPerInstance`, fronted by API Gateway/Lambda.
- **Result:** A managed, autoscaling, monitored inference API with no servers to patch.

#### Use Case 3: Nightly batch scoring of a large dataset

- **Context:** Score 200 GB of records once a day; no online latency requirement.
- **Implementation:** **Batch Transform** job reads the S3 dataset across N instances, writes
  predictions back to S3, and the cluster is destroyed on completion.
- **Result:** Pay only for the minutes the batch runs; no endpoint idling overnight.

#### Use Case 4: Governed train→deploy pipeline

- **Context:** A regulated team needs reproducible, approved model promotions.
- **Implementation:** A **SageMaker Pipeline** (process → train → evaluate → conditionally
  register) plus **Model Registry** approval gating an automated deploy, with **Clarify** bias
  reports and **Model Monitor** drift alarms.
- **Result:** Every production model is versioned, explained, approved, and watched for drift.

## Best Practices

<a id='best-practices'></a>

1. **Always delete endpoints (and stop notebook instances).** Real-time endpoints and notebook
   instances bill per second whether or not they serve traffic — `predictor.delete_endpoint()`
   is the single most important cost habit.
2. **Use Managed Spot Training with checkpointing** for anything interruption-tolerant; it's a
   near-free ~70–90% saving once `max_wait` + `checkpoint_s3_uri` are set.
3. **Lean on framework estimators and prebuilt DLCs** before writing a custom container — bring
   `train.py` + `requirements.txt`, not a Dockerfile.
4. **Choose the right `TrainingInputMode`.** `File` copies all data first (simple, small data);
   `FastFile`/`Pipe` stream from S3 (large datasets, faster start).
5. **Scope the IAM execution role tightly** to the specific S3 buckets and ECR repos it needs,
   rather than blanket `AmazonSageMakerFullAccess` in production.
6. **Match the inference shape to the SLA:** real-time for steady low-latency, serverless for
   spiky/low traffic, async for large payloads/long jobs, batch for offline scoring.
7. **Enable autoscaling on endpoints** with a target-tracking policy on
   `InvocationsPerInstance` so you don't over- or under-provision.
8. **Version everything through the Model Registry and Pipelines** so deployments are
   reproducible and approvals are auditable.

## Common Pitfalls

<a id='pitfalls'></a>

1. **Forgotten endpoints draining money.** A left-running `ml.g5.2xlarge` endpoint costs ~$1+/hr
   indefinitely. *Avoid by* deleting endpoints in a `finally`, and alarming on
   `Endpoints in service`.
2. **IAM "AccessDenied" on S3/ECR.** The execution role lacks `s3:GetObject` on the data bucket
   or `ecr:GetDownloadUrlForLayer` on a custom image. *Avoid by* granting the role explicit
   access to exactly those resources.
3. **Spot job restarts from scratch.** Without `checkpoint_s3_uri`, a reclaimed Spot instance
   loses all progress. *Avoid by* checkpointing to `/opt/ml/checkpoints` (synced to S3).
4. **Saving the model to the wrong directory.** Only what's written to `SM_MODEL_DIR`
   (`/opt/ml/model`) ends up in `model.tar.gz`. *Avoid by* always saving there.
5. **Service quotas block GPU jobs.** New accounts have a `0` quota for many `ml.p*`/`ml.g*`
   instances. *Avoid by* requesting quota increases before you need the capacity.
6. **Over-provisioned instances.** Picking `ml.p4d.24xlarge` for a tiny model wastes money.
   *Avoid by* right-sizing and using Inference Recommender / load tests.
7. **Region/bucket mismatch.** Training data, endpoint, and bucket must share a region or you
   pay egress and hit latency. *Avoid by* keeping the Session region consistent.
8. **Treating Studio compute as free.** Studio kernels and apps run on real instances that keep
   billing until shut down. *Avoid by* setting idle-shutdown and stopping unused apps.

## Performance Optimization

<a id='performance'></a>

### Configuration Tuning

Key parameters to optimize:

- **`instance_type` / `instance_count`** — the core throughput-vs-cost lever. Scale to multi-GPU
  (`ml.p4d`/`ml.p5`) and multi-node only when a single GPU is the bottleneck; communication
  overhead means scaling efficiency falls off, so measure it.
- **`TrainingInputMode`** — `FastFile`/`Pipe` start training before all data is copied and avoid
  a giant EBS volume; for large datasets they cut both startup time and cost vs `File`.
- **Distributed strategy** — SMDDP's optimized AllReduce beats vanilla NCCL on AWS networking;
  for huge models, SMP/FSDP sharding fits parameters that don't fit one GPU.
- **Mixed precision & compilation** — `bf16`/`fp16` and SageMaker **Training Compiler** (or
  `torch.compile`) raise GPU throughput substantially.
- **Endpoint sizing & autoscaling** — set the autoscaling target on `InvocationsPerInstance`
  from a load test; use **multi-model** or **multi-container** endpoints to pack many small
  models onto one instance, and **Inference Recommender** to find the cheapest type meeting your
  latency SLA.

In [ ]:
# Estimate training cost and scaling efficiency to decide how many instances to use.
# On-demand $/hr are illustrative; check the SageMaker pricing page for your region.
PRICE_PER_HR = {"ml.g5.2xlarge": 1.52, "ml.p3.2xlarge": 3.83, "ml.p4d.24xlarge": 37.69}

def training_cost(instance_type, instance_count, hours, scaling_efficiency=0.8, spot=False):
    rate = PRICE_PER_HR[instance_type]
    if spot:
        rate *= 0.30                                   # ~70% Spot discount
    # More nodes finish faster, but with sub-linear speedup from comms overhead.
    effective_speedup = 1 + (instance_count - 1) * scaling_efficiency
    wall_hours = hours / effective_speedup
    cost = rate * instance_count * wall_hours
    return wall_hours, cost

base_hours = 10  # single-instance training time
print(f"{'config':<34}{'wall (h)':>10}{'cost ($)':>12}")
for n in (1, 2, 4):
    for spot in (False, True):
        w, c = training_cost("ml.p4d.24xlarge", n, base_hours, spot=spot)
        tag = f"p4d x{n}{' spot' if spot else ''}"
        print(f"{tag:<34}{w:>10.2f}{c:>12.2f}")
# Note: 4x instances finish ~3.4x faster (not 4x) due to 0.8 scaling efficiency.

## Production Deployment

<a id='deployment'></a>

### Real-time endpoint with autoscaling (boto3 / Application Auto Scaling)

A model is deployed in three API objects: **Model** (artifact + image) → **EndpointConfig**
(instance type, count, variants) → **Endpoint**. Autoscaling is attached to the variant:

```python
import boto3
aas = boto3.client("application-autoscaling")
resource_id = "endpoint/my-endpoint/variant/AllTraffic"

aas.register_scalable_target(
    ServiceNamespace="sagemaker", ResourceId=resource_id,
    ScalableDimension="sagemaker:variant:DesiredInstanceCount",
    MinCapacity=1, MaxCapacity=8,
)
aas.put_scaling_policy(
    PolicyName="invocations-target", ServiceNamespace="sagemaker",
    ResourceId=resource_id,
    ScalableDimension="sagemaker:variant:DesiredInstanceCount",
    PolicyType="TargetTrackingScaling",
    TargetTrackingScalingPolicyConfiguration={
        "TargetValue": 750.0,   # invocations per instance per minute
        "PredefinedMetricSpecification": {
            "PredefinedMetricType": "SageMakerVariantInvocationsPerInstance"},
        "ScaleInCooldown": 300, "ScaleOutCooldown": 60,
    },
)
```

### Safe rollouts: blue/green and canary

`update_endpoint` supports **blue/green** deployment with a **canary** traffic shift and
CloudWatch alarms that auto-roll-back, so a bad model version never takes 100% of traffic:

```python
DeploymentConfig = {
  "BlueGreenUpdatePolicy": {
    "TrafficRoutingConfiguration": {
      "Type": "CANARY", "CanarySize": {"Type": "CAPACITY_PERCENT", "Value": 10},
      "WaitIntervalInSeconds": 300},
    "TerminationWaitInSeconds": 600},
  "AutoRollbackConfiguration": {"Alarms": [{"AlarmName": "endpoint-5xx-high"}]},
}
```

### Serverless for spiky traffic

```python
from sagemaker.serverless import ServerlessInferenceConfig
predictor = model.deploy(
    serverless_inference_config=ServerlessInferenceConfig(
        memory_size_in_mb=4096, max_concurrency=20))
```

### Infrastructure as code

For repeatable environments, define endpoints with **CloudFormation/CDK** or wire deployment
into a **SageMaker Pipeline** triggered by a Model Registry approval, rather than clicking in the
console.

## Monitoring and Observability

<a id='monitoring'></a>

### Key Metrics to Track

- **`Invocations`, `InvocationsPerInstance`** — traffic and the basis for autoscaling.
- **`ModelLatency`, `OverheadLatency`** — time in your model vs SageMaker overhead; the split
  tells you whether to optimize the model or the serving stack.
- **`Invocation4XXErrors` / `Invocation5XXErrors`** — client vs server failures; 5XX spikes
  often mean OOM or a crashed model server.
- **`CPUUtilization`, `GPUUtilization`, `GPUMemoryUtilization`, `MemoryUtilization`** — endpoint
  and training-instance saturation; drives right-sizing.
- **Training job metrics** — emit `loss`/`accuracy` via `metric_definitions` regexes so they
  surface in CloudWatch and the console.

### Tooling

- **CloudWatch** — all endpoint/training metrics and logs land here automatically; build alarms
  (e.g., 5XX rate, latency p99) and dashboards from them.
- **SageMaker Model Monitor** — scheduled jobs that compare live traffic against a baseline to
  detect **data drift**, **model quality** decay, bias drift, and feature attribution drift.
- **SageMaker Clarify** — bias and SHAP-based explainability reports for training and inference.
- **Debugger / Profiler** — captures tensors and system metrics during training to catch
  vanishing gradients, GPU under-utilization, and bottlenecks.

### Logging Best Practices

- Log to **stdout/stderr** in your container — SageMaker ships it to CloudWatch Logs under
  `/aws/sagemaker/...`.
- Tag jobs and endpoints (team, project, env) so cost and logs are attributable.
- Capture endpoint request/response with **Data Capture** to S3 to feed Model Monitor and to
  debug production predictions.

## Troubleshooting

<a id='troubleshooting'></a>

#### Issue 1: Training job fails with `AccessDenied` reading S3

**Symptoms:** Job moves to `Failed` almost immediately; logs show `AccessDenied` /
`403` fetching the input channel.

**Cause:** The **IAM execution role** lacks `s3:GetObject`/`s3:ListBucket` on the data bucket
(or the bucket policy / KMS key denies it).

**Solution:** Add an explicit policy granting the role access to the exact bucket/prefix (and
the KMS key if the bucket is encrypted); confirm the bucket is in the job's region.

#### Issue 2: `ResourceLimitExceeded` when launching a GPU job

**Symptoms:** `CreateTrainingJob`/`CreateEndpoint` fails with a quota error for
`ml.p4d.24xlarge` (or similar).

**Cause:** Your account's **service quota** for that instance type in that region is `0` or too
low.

**Solution:** Request a quota increase via Service Quotas for the specific instance type and
region, then retry. Test logic on a smaller available type first.

#### Issue 3: Endpoint returns 5XX or won't reach `InService`

**Symptoms:** Invocations fail with `ModelError`/5XX, or the endpoint sticks in `Creating` then
`Failed`.

**Cause:** The model server crashed (bad artifact, missing dependency, OOM) or the inference
handler (`model_fn`/`predict_fn`) raised an exception on load/inference.

**Solution:** Read the endpoint's CloudWatch logs, reproduce locally with **SageMaker Local
Mode**, verify `model.tar.gz` layout and `requirements.txt`, and right-size memory if it's OOM.

#### Issue 4: Spot training keeps restarting / never finishes

**Symptoms:** Job repeatedly interrupts and resumes from epoch 0; total time balloons.

**Cause:** Frequent Spot interruptions with no checkpointing, so each restart loses progress.

**Solution:** Set `checkpoint_s3_uri`, write/resume checkpoints from `/opt/ml/checkpoints`, and
raise `max_wait`; consider on-demand for very interruption-sensitive runs.

## Comparison with Alternatives

<a id='comparison'></a>

### How SageMaker Compares to Other ML Platforms

| Dimension | Amazon SageMaker | Self-managed K8s (Kubeflow) | Vertex AI (GCP) / Azure ML |
|-----------|------------------|-----------------------------|----------------------------|
| Cloud | AWS-native | Any cloud / on-prem | GCP / Azure native |
| Ops burden | Low — fully managed jobs & endpoints | High — you run the cluster | Low — managed |
| Flexibility | Medium — AWS conventions, BYO container | High — full control | Medium |
| Distributed training | Built-in SMDDP/SMP + torchrun/MPI | DIY (operators, MPI) | Built-in |
| Inference options | Real-time, serverless, async, batch | DIY (KServe, Triton) | Managed online/batch |
| Lifecycle tooling | Pipelines, Registry, Feature Store, Monitor, Clarify | Assemble from OSS | Comparable managed suite |
| Lock-in | AWS APIs & artifact formats | Portable / open | Cloud-specific |
| Cost model | Per-second instances; pay only when running | Pay for standing cluster + your time | Per-second managed |

### When to Choose SageMaker

- You're committed to **AWS** and want managed training/serving without running Kubernetes.
- You value the **integrated lifecycle** (pipelines, registry, monitoring) over maximum
  framework freedom.
- Your GPU needs are **bursty** — pay per second instead of owning idle hardware.
- You can accept **AWS-specific conventions and some lock-in** in exchange for lower ops burden.

Prefer self-managed Kubeflow/Ray/KServe when you need multi-cloud portability or deep cluster
control; prefer Vertex AI / Azure ML if your stack already lives on GCP or Azure.

## Resources

<a id='resources'></a>

### Official Documentation

- Amazon SageMaker Developer Guide: <https://docs.aws.amazon.com/sagemaker/latest/dg/whatis.html>
- SageMaker Python SDK: <https://sagemaker.readthedocs.io/>
- Deep Learning Containers (DLC) image URIs: <https://github.com/aws/deep-learning-containers/blob/master/available_images.md>

### Tutorials and Guides

- Amazon SageMaker Examples (GitHub): <https://github.com/aws/amazon-sagemaker-examples>
- Distributed Training on SageMaker: <https://docs.aws.amazon.com/sagemaker/latest/dg/distributed-training.html>
- Managed Spot Training: <https://docs.aws.amazon.com/sagemaker/latest/dg/model-managed-spot-training.html>

### Community Resources

- AWS Machine Learning Blog: <https://aws.amazon.com/blogs/machine-learning/>
- re:Post (AWS Q&A): <https://repost.aws/tags/TAQz4M4dWcS5q5l2u2qg1z-Q/amazon-sage-maker>
- Stack Overflow tag: <https://stackoverflow.com/questions/tagged/amazon-sagemaker>

### Related Technologies

- **AWS Deep Learning Containers** — the optimized training/inference images SageMaker uses.
- **Amazon Bedrock** — managed access to foundation models (complementary to SageMaker hosting).
- **AWS Step Functions / SageMaker Pipelines** — orchestration for ML workflows.
- **Amazon S3, ECR, IAM, CloudWatch** — the storage, registry, identity, and observability
  substrate every SageMaker job relies on.